In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import segmentation_models_pytorch as smp
import torch.optim as optim
import torch.nn as nn
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
from torchsummary import summary

In [2]:
def rgb_to_class(mask):
    mask = np.array(mask)
    mask = mask * 255
    mask = mask.astype(np.uint8)

    height, width = mask.shape[1], mask.shape[2]
    
    mask_out = np.zeros((8, height, width), dtype=np.uint8)
    
    class_colors = {
        0: [0, 0, 0],
        1: [128, 0, 128],
        2: [173, 216, 230],
        3: [255, 0, 0],
        4: [255, 255, 0],
        5: [0, 255, 0],
        6: [0, 0, 255],
        7: [255, 255, 255]
    }

    mask_flat = mask.reshape(3, -1).T 
    colors_flat = np.array(list(class_colors.values())) 
    distances = np.linalg.norm(mask_flat[:, None] - colors_flat, axis=2)
    closest_color_indices = np.argmin(distances, axis=1) 
    closest_color_indices = closest_color_indices.reshape((height, width))

    
    for i in range(8):
        mask_out[i] = (closest_color_indices == i).astype(np.uint8)
    return mask_out

def class_to_rgb(mask):
    
    _, height, width = mask.shape
    
    
    rgb_mask = np.zeros((height, width, 3), dtype=np.uint8)
    
    
    class_colors = {
        0: [0, 0, 0],
        1: [128, 0, 128],
        2: [173, 216, 230],
        3: [255, 0, 0],
        4: [255, 255, 0],
        5: [0, 255, 0],
        6: [0, 0, 255],
        7: [255, 255, 255]
    }

    for class_idx, color in class_colors.items():
        
        rgb_mask[mask[class_idx] == 1] = color

    return rgb_mask

def convert_rgb(mask):
    
    print(mask.shape, np.unique(mask))
    height, width = mask.shape
    rgb_mask = np.zeros((height, width, 3), dtype=np.uint8)

    rgb_mask[mask == 0] = [0, 0, 0]
    rgb_mask[mask == 1] = [128, 0, 128]
    rgb_mask[mask == 2] = [173, 216, 230]
    rgb_mask[mask == 3] = [255, 0, 0]
    rgb_mask[mask == 4] = [255, 255, 0]
    rgb_mask[mask == 5] = [0, 255, 0]
    rgb_mask[mask == 6] = [0, 0, 255]
    rgb_mask[mask == 7] = [255, 255, 255]

    return rgb_mask



class model_segment:
    def __init__(self, check_point_path):
        self.model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=8
        )
        self.transform = transforms.Compose([
            transforms.Resize((192, 320)),
            transforms.ToTensor(),
        ])
        self.loss_fn = loss_fn = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-4)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        
        
        checkpoint_path = check_point_path
        checkpoint = torch.load(checkpoint_path)

        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])



    def check(self):
        image = Image.open(r"D:\Demo 1.2\Unet\dataset\train\input\00905.jpg").convert('RGB')
        image = self.transform(image).unsqueeze(0).to(self.device)
        print(image.shape)
        
        self.model.eval()
        with torch.no_grad():
            output = self.model(image)
        max_indices = torch.argmax(output, dim=1)

        k = 0
        for i in range(192):
            for j in range(320):
                pixel_values = output[0, :, i, j]

                argmax_value = torch.argmax(pixel_values)
                
                if argmax_value != 0 and argmax_value != 1:
                    k += 1
                    print(f"Pixel ({i}, {j}) has argmax value: {argmax_value}")
                    break
            if k == 1:
                break
        if k == 0:
            print("No pixel found with an unexpected class")


    def load_data(self, input_path, output_path):
        
        image_dir = input_path
        mask_dir = output_path


        train_dataset = CustomRGBDataset(image_dir, mask_dir, transform=self.transform)
        self.train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


    def train(self, num_epochs, checkpoint_out):

        for epoch in range(num_epochs):
            self.model.train()
            train_loss = 0

            for images, masks in self.train_loader:


                images = images.to(self.device)
                masks = torch.argmax(masks, dim=1).to(self.device)


                outputs = self.model(images)

                loss = self.loss_fn(outputs, masks)


                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                train_loss += loss.item()
            self.check()
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss/len(self.train_loader)}")

            checkpoint_path = checkpoint_out
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'loss': train_loss / len(self.train_loader),
            }, checkpoint_path)
            print(f"Checkpoint saved at {checkpoint_path}")


    def test(self, image):
        image = self.transform(image).unsqueeze(0).to(self.device)

        self.model.eval()
        with torch.no_grad():
            output = self.model(image)

        max_indices = torch.argmax(output, dim=1)


        max_indices = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
        pred_mask_rgb = convert_rgb(max_indices)

        return pred_mask_rgb
                



class CustomRGBDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(('.png', '.jpg', '.jpeg'))])
        self.mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) if fname.endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transforms.Compose([
            transforms.Resize((192, 320)),
            transforms.ToTensor(),
        ])
            
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        mask = Image.open(self.mask_paths[idx]).convert('RGB')

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        

        mask = rgb_to_class(mask)

        return image, mask



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


model = model_segment('checkpoint.pth')
model.load_data(input_path = 'dataset/train/input',output_path= 'dataset/train/output')
model.train(num_epochs = 5, checkpoint_out= "checkpoint1.pth")


path = "dataset/test/input"
images = os.listdir(path)
for image in images:
    path_image = path + "/" + image

    image = Image.open(path_image).convert('RGB')
    print(type(image))
    image = model.test(image)

    plt.imshow(image)
    plt.axis('off')
    plt.show()



